# 02 - Preprocessing Walkthrough

This notebook walks through the data preprocessing pipeline step by step:
1. Load raw data
2. Clean and filter (minimum rating thresholds)
3. Build movie metadata
4. Temporal train/test split
5. Feature engineering (user-item matrix, content features)

The code mirrors `src/data/preprocess.py` and `src/data/feature_engineering.py`,
but is presented interactively with visualisations of before/after effects.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
from pathlib import Path

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

RAW_DIR = Path("../data/raw")
PROC_DIR = Path("../data/processed")

# Thresholds from params.yaml
MIN_USER_RATINGS = 20
MIN_MOVIE_RATINGS = 50
TEST_RATIO = 0.2

## 1. Load Raw Data

In [ ]:
ratings_raw = pd.read_csv(RAW_DIR / "ratings.csv")
movies_raw = pd.read_csv(RAW_DIR / "movies.csv")
links_raw = pd.read_csv(RAW_DIR / "links.csv")
tags_raw = pd.read_csv(RAW_DIR / "tags.csv")

print(f"Raw ratings: {ratings_raw.shape}")
print(f"Raw movies:  {movies_raw.shape}")
print(f"Raw links:   {links_raw.shape}")
print(f"Raw tags:    {tags_raw.shape}")
display(ratings_raw.head())

## 2. Cleaning: Minimum Rating Thresholds

We iteratively filter users with fewer than 20 ratings and movies with fewer than 50 ratings.
This removes noise from infrequent users and obscure movies.

In [ ]:
def apply_min_thresholds(df, min_user=20, min_movie=50, max_iter=10):
    """Iteratively filter until stable."""
    history = [{"users": df["userId"].nunique(), "movies": df["movieId"].nunique(), "ratings": len(df)}]
    
    for i in range(max_iter):
        before = len(df)
        # Filter users
        user_counts = df.groupby("userId").size()
        valid_users = user_counts[user_counts >= min_user].index
        df = df[df["userId"].isin(valid_users)]
        
        # Filter movies
        movie_counts = df.groupby("movieId").size()
        valid_movies = movie_counts[movie_counts >= min_movie].index
        df = df[df["movieId"].isin(valid_movies)]
        
        history.append({"users": df["userId"].nunique(), "movies": df["movieId"].nunique(), "ratings": len(df)})
        
        if len(df) == before:
            print(f"Converged after {i+1} iterations")
            break
    
    return df, pd.DataFrame(history)

ratings_clean, filter_history = apply_min_thresholds(
    ratings_raw.copy(), MIN_USER_RATINGS, MIN_MOVIE_RATINGS
)

print("\nFiltering history:")
display(filter_history)

In [ ]:
# Before vs After visualisation
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

labels = ["Before", "After"]
values_users = [ratings_raw["userId"].nunique(), ratings_clean["userId"].nunique()]
values_movies = [ratings_raw["movieId"].nunique(), ratings_clean["movieId"].nunique()]
values_ratings = [len(ratings_raw), len(ratings_clean)]

for ax, vals, title, color in [
    (axes[0], values_users, "Users", "#2196F3"),
    (axes[1], values_movies, "Movies", "#4CAF50"),
    (axes[2], values_ratings, "Ratings", "#FF9800"),
]:
    bars = ax.bar(labels, vals, color=[color, color], alpha=[0.5, 1.0], edgecolor="white")
    ax.set_title(title)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{val:,}",
                ha="center", va="bottom", fontsize=11)
    pct_kept = vals[1] / vals[0] * 100
    ax.set_ylabel(f"Count ({pct_kept:.1f}% retained)")

plt.suptitle("Effect of Minimum Rating Thresholds", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Movie Metadata

Build a metadata table with title, genres, year, and average rating.

In [ ]:
# Extract year from title
movies_meta = movies_raw.copy()
movies_meta["year"] = movies_meta["title"].str.extract(r"\((\d{4})\)$")
movies_meta["year"] = pd.to_numeric(movies_meta["year"], errors="coerce")
movies_meta["clean_title"] = movies_meta["title"].str.replace(r"\s*\(\d{4}\)$", "", regex=True)

# Add aggregate rating info
movie_stats = ratings_clean.groupby("movieId").agg(
    avg_rating=("rating", "mean"),
    n_ratings=("rating", "count")
).reset_index()

movies_meta = movies_meta.merge(movie_stats, on="movieId", how="inner")
movies_meta = movies_meta.merge(links_raw[["movieId", "tmdbId"]], on="movieId", how="left")

print(f"Movie metadata shape: {movies_meta.shape}")
display(movies_meta.head(10))

## 4. Temporal Train/Test Split

For each user, sort ratings chronologically and assign the last 20% to the test set.
This simulates the real-world scenario where we predict future preferences.

In [ ]:
def temporal_split(df, test_ratio=0.2):
    """Per-user temporal split."""
    train_parts = []
    test_parts = []
    
    for uid, group in df.groupby("userId"):
        group_sorted = group.sort_values("timestamp")
        n_test = max(1, int(len(group_sorted) * test_ratio))
        train_parts.append(group_sorted.iloc[:-n_test])
        test_parts.append(group_sorted.iloc[-n_test:])
    
    return pd.concat(train_parts), pd.concat(test_parts)

train_df, test_df = temporal_split(ratings_clean, TEST_RATIO)

print(f"Train set: {train_df.shape}  ({len(train_df)/len(ratings_clean)*100:.1f}%)")
print(f"Test set:  {test_df.shape}  ({len(test_df)/len(ratings_clean)*100:.1f}%)")

In [ ]:
# Visualise temporal split for a sample user
sample_user = train_df["userId"].value_counts().index[0]
user_train = train_df[train_df["userId"] == sample_user].sort_values("timestamp")
user_test = test_df[test_df["userId"] == sample_user].sort_values("timestamp")

fig, ax = plt.subplots(figsize=(14, 4))
train_dates = pd.to_datetime(user_train["timestamp"], unit="s")
test_dates = pd.to_datetime(user_test["timestamp"], unit="s")

ax.scatter(train_dates, user_train["rating"], c="steelblue", label="Train", alpha=0.7, s=20)
ax.scatter(test_dates, user_test["rating"], c="red", label="Test", alpha=0.7, s=20)
ax.axvline(train_dates.iloc[-1], color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Date")
ax.set_ylabel("Rating")
ax.set_title(f"Temporal Split for User {sample_user} ({len(user_train)} train, {len(user_test)} test)")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Feature Matrix Statistics

After preprocessing, examine the user-item matrix and content features.

In [ ]:
# Load processed artifacts (if they exist)
try:
    ui_matrix = sp.load_npz(PROC_DIR / "user_item_matrix.npz")
    content_features = sp.load_npz(PROC_DIR / "content_features.npz")
    
    print("User-Item Matrix")
    print(f"  Shape: {ui_matrix.shape}")
    print(f"  Non-zero: {ui_matrix.nnz:,}")
    print(f"  Density: {ui_matrix.nnz / (ui_matrix.shape[0] * ui_matrix.shape[1]):.4%}")
    print(f"  Dtype: {ui_matrix.dtype}")
    
    print(f"\nContent Features")
    print(f"  Shape: {content_features.shape}")
    print(f"  Non-zero: {content_features.nnz:,}")
    print(f"  Density: {content_features.nnz / (content_features.shape[0] * content_features.shape[1]):.4%}")
    
    # Row-wise density of content features
    row_nnz = np.diff(content_features.indptr)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(row_nnz, bins=50, color="steelblue", edgecolor="white")
    axes[0].set_xlabel("Non-zero features per movie")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Content Feature Density per Movie")
    
    # Distribution of non-zero values in user-item matrix
    axes[1].hist(ui_matrix.data, bins=9, color="steelblue", edgecolor="white")
    axes[1].set_xlabel("Rating Value")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Rating Values in User-Item Matrix")
    
    plt.tight_layout()
    plt.show()
    
except FileNotFoundError:
    print("Processed files not found. Run the preprocessing pipeline first:")
    print("  python -m src.data.preprocess")
    print("  python -m src.data.feature_engineering")

In [ ]:
print("Preprocessing walkthrough complete.")